In [ ]:
resnet的使用.py
'''
 MNIST包含70,000张手写数字图像: 60,000张用于训练，10,000张用于测试。
 图像是灰度的，28x28像素的，并且居中的，以减少预处理和加快运行。
'''
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchvision.models import resnet18
import torch.nn.functional as F

'''下载训练数据集（包含训练图片+标签）'''
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),#张量
)   #对于pytorch库能够识别的数据一般是tensor张力
# datasets.MNIST的参数：
#   root(string)： 表示数据集的根目录，
#   train(bool, optional)： 如果为True，则从training.pt创建数据集，否则从test.pt创建数据集
#   download(bool, optional)： 如果为True，则从internet下载数据集并将其放入根目录。如果数据集已下载，则不会再次下载
#   transform(callable, optional)： 接收PIL图片并返回转换后版本图片的转换函数

'''下载测试数据集（包含训练图片+标签） '''
test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)
# print(len(training_data))

'''展示手写字图片，把训练数据集中的前9张图片展示一下'''
from matplotlib import pyplot as plt
figure = plt.figure()
for i in range(9):
    img, label = training_data[i]#0
    figure.add_subplot(3, 3, i+1)
    plt.title(label)
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")
plt.show()

'''创建数据DataLoader（数据加载器）
    batch_size:将数据集分成多份，每一份为batch_size个数据。
           优点：可以减少内存的使用，提高训练速度。
'''
train_dataloader = DataLoader(training_data, batch_size=64)#64张图片为一个包，
test_dataloader = DataLoader(test_data, batch_size=64)
for X, y in test_dataloader:#X是表示打包好的每一个数据包
    print(f"Shape of X [N, C, H, W]: {X.shape}")#
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

'''判断当前设备是否支持GPU，其中mps是苹果m系列芯片的GPU。'''
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using {device} device")

''' 定义神经网络  '''
#模块搭建
class ResBlock(nn.Module):
    def __init__(self,channels_in):
        super().__init__()
        self.conv1=torch.nn.Conv2d(channels_in,30,5,padding=2)
        self.conv2=torch.nn.Conv2d(30,channels_in,3,padding=1)

    def forward(self,x):
        out=self.conv1(x)
        out=self.conv2(out)
        return F.relu(out+x)

#网络搭建
class ResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=torch.nn.Conv2d(1,20,5)
        self.conv2=torch.nn.Conv2d(20,15,3)
        self.maxpool=torch.nn.MaxPool2d(2)
        self.resblock1=ResBlock(channels_in=20)
        self.resblock2=ResBlock(channels_in=15)
        self.full_c=torch.nn.Linear(375,10)

    def forward(self,x):
        size=x.shape[0]
        x=F.relu(self.maxpool(self.conv1(x)))
        x=self.resblock1(x)
        x=F.relu(self.maxpool(self.conv2(x)))
        x=self.resblock2(x)
        x=x.view(size,-1)
        x=self.full_c(x)
        return x

model=ResNet()

# model = resnet18(num_classes=19)
print(model)
model = model.to(device)

def train(dataloader, model, loss_fn, optimizer):
    model.train()
#pytorch提供2种方式来切换训练和测试的模式，分别是：model.train() 和 model.eval()。
# 一般用法是：在训练开始之前写上model.trian()，在测试时写上 model.eval() 。
    batch_size_num = 1
    for X, y in dataloader:                 #其中batch为每一个数据的编号
        X, y = X.to(device), y.to(device)   #把训练数据集和标签传入cpu或GPU
        pred = model.forward(X)                     #自动初始化 w权值
        loss = loss_fn(pred, y)             #通过交叉熵损失函数计算损失值loss
        # Backpropagation 进来一个batch的数据，计算一次梯度，更新一次网络
        optimizer.zero_grad()               #梯度值清零
        loss.backward()                     #反向传播计算得到每个参数的梯度值
        optimizer.step()                    #根据梯度更新网络参数

        loss = loss.item()                  #获取损失值
        print(f"loss: {loss:>7f}  [number:{batch_size_num}]")
        batch_size_num += 1

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()    #
    test_loss, correct = 0, 0
    with torch.no_grad():   #一个上下文管理器，关闭梯度计算。当你确认不会调用Tensor.backward()的时候。这可以减少计算所用内存消耗。
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model.forward(X)
            test_loss += loss_fn(pred, y).item() #
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            a = (pred.argmax(1) == y)  #dim=1表示每一行中的最大值对应的索引号，dim=0表示每一列中的最大值对应的索引号
            b = (pred.argmax(1) == y).type(torch.float)
    test_loss /= num_batches
    correct /= size
    print(f"Test result: \n Accuracy: {(100*correct)}%, Avg loss: {test_loss}")

loss_fn = nn.CrossEntropyLoss() #创建交叉熵损失函数对象，因为手写字识别中一共有10个数字，输出会有10个结果
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)#创建一个优化器，SGD为随机梯度下降算法？？
#params：要训练的参数，一般我们传入的都是model.parameters()。
#lr：learning_rate学习率，也就是步长。


train(train_dataloader, model, loss_fn, optimizer)
test(test_dataloader, model, loss_fn)


# epochs = 10 #
# for t in range(epochs):
#     print(f"Epoch {t+1}\n-------------------------------")
#     train(train_dataloader, model, loss_fn, optimizer)
#     # test(test_dataloader, model, loss_fn)
# print("Done!")
# test(test_dataloader, model, loss_fn)
#
#
#
# #分析sigmiod，relu
# # sgd，Adam